# Hand Gesture Recognition: training notebook

You will: download a HaGRID sample, crop the hands, fine-tune a pretrained **MobileNetV3**, evaluate it, and save `gesture_model.pt` for the live webcam app (`app.py`).

**Before you start**
1. Colab menu: *Runtime > Change runtime type > T4 GPU*.
2. Kaggle account: *Settings > API > Create New Token* downloads `kaggle.json` (you'll upload it in Step 2).

## Step 1: Check the GPU
Training on a GPU is 10-30x faster than on a CPU.

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## Step 2: Download the dataset
We use a ~31k-image, 384p sample of **HaGRID** (RGB webcam-style photos, many different people and backgrounds).
Check the dataset's license on its Kaggle page before using it beyond learning.

In [ ]:
from google.colab import files
files.upload()   # pick your kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d innominate817/hagrid-sample-30k-384p -p /content/hagrid --unzip

In [ ]:
from pathlib import Path
ROOT = Path("/content/hagrid")
ANN_DIR = next(ROOT.rglob("ann_train_val"))   # JSON annotations (hand boxes + labels)
IMG_DIR = next(ROOT.rglob("hagrid_30k"))      # images, one folder per gesture
print(ANN_DIR)
print(IMG_DIR)
print(sorted(p.name for p in IMG_DIR.iterdir()))

## Step 3: Crop the hands and split by person
Why crop? At webcam time we will find the hand first and give the model only that crop, so we train the same way.

Why split by person? Photos of the same person look alike. If the same person is in both train and test, accuracy looks fake-high. We use each image's `user_id` so a person lands in only one split (about 70% train / 15% val / 15% test).

Edit `CLASSES` to choose gestures. Start with a few distinct ones.

In [ ]:
import json, hashlib
from PIL import Image

CLASSES = ["call", "dislike", "fist", "like", "ok", "one", "palm", "peace", "rock", "stop"]
PAD = 0.25                  # margin around the hand (fraction of its size). Keep the same in app.py
OUT = Path("/content/crops")

def split_of(user_id):
    h = int(hashlib.md5(user_id.encode()).hexdigest(), 16) % 100
    return "train" if h < 70 else "val" if h < 85 else "test"

def square_crop(img, box, pad=PAD):
    W, H = img.size
    x, y, w, h = box[0] * W, box[1] * H, box[2] * W, box[3] * H   # boxes are fractions of image size
    cx, cy = x + w / 2, y + h / 2
    side = max(w, h) * (1 + 2 * pad)
    return img.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2)))

counts = {"train": 0, "val": 0, "test": 0}
for g in CLASSES:
    ann = json.load(open(ANN_DIR / f"{g}.json"))
    folder = IMG_DIR / f"train_val_{g}"
    for img_id, a in ann.items():
        path = folder / f"{img_id}.jpg"
        if not path.exists():
            continue
        for box, label in zip(a["bboxes"], a["labels"]):
            if label != g:          # skip the extra "no_gesture" hand
                continue
            split = split_of(a["user_id"])
            dest = OUT / split / g
            dest.mkdir(parents=True, exist_ok=True)
            crop = square_crop(Image.open(path).convert("RGB"), box).resize((224, 224))
            crop.save(dest / f"{img_id}.jpg", quality=95)
            counts[split] += 1
            break
print(counts)

## Step 4: Load the data (with augmentation)
Augmentation randomly rotates, zooms, flips, and changes colors during training so the model copes with real-world variation. Validation/test images are only resized.

Horizontal flips are fine here because HaGRID has left and right hands. Remove the flip if you add a gesture where left vs right matters.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats the pretrained model expects

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tfm = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.ImageFolder(OUT / "train", train_tfm)   # folder name = label
val_ds   = datasets.ImageFolder(OUT / "val", eval_tfm)
test_ds  = datasets.ImageFolder(OUT / "test", eval_tfm)
classes = train_ds.classes
print(classes, len(train_ds), len(val_ds), len(test_ds))

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, batch_size=64, num_workers=2)
test_dl  = DataLoader(test_ds, batch_size=64, num_workers=2)

## Step 5: Load the pretrained model
`weights="DEFAULT"` downloads MobileNetV3 already trained on ImageNet (1.2M images). It already understands edges, shapes and textures. We only replace its last layer so it outputs *our* gestures.

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.mobilenet_v3_large(weights="DEFAULT")
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model = model.to(device)

## Step 6: Fine-tune in two phases
- **Phase 1:** freeze the pretrained part (`features`) and train only the new last layers. Quick and stable.
- **Phase 2:** unfreeze everything and train gently (small learning rate) so the model adapts to hands.

Watch `val_acc`. If `train_acc` keeps rising while `val_acc` stalls, the model is overfitting.

In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total = correct = loss_sum = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item()
            total += len(y)
    return loss_sum / total, correct / total

best_acc = 0.0
def fit(epochs, optimizer):
    global best_acc
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(train_dl, optimizer)
        va_loss, va_acc = run_epoch(val_dl)
        print(f"epoch {ep}: train_acc {tr_acc:.3f}  val_acc {va_acc:.3f}  val_loss {va_loss:.3f}")
        if va_acc > best_acc:                       # keep the best version
            best_acc = va_acc
            torch.save(model.state_dict(), "best.pt")

# Phase 1: only the new head learns
for p in model.features.parameters():
    p.requires_grad = False
opt1 = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)
fit(3, opt1)

# Phase 2: everything learns, gently
for p in model.parameters():
    p.requires_grad = True
opt2 = torch.optim.Adam(model.parameters(), lr=1e-4)
fit(8, opt2)

## Step 7: Test on people the model has never seen
The confusion matrix shows which gestures get mixed up (rows = true, columns = predicted).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

model.load_state_dict(torch.load("best.pt", map_location=device))
model.eval()
preds, truth = [], []
with torch.no_grad():
    for x, y in test_dl:
        preds += model(x.to(device)).argmax(1).cpu().tolist()
        truth += y.tolist()

print("Test accuracy:", np.mean(np.array(preds) == np.array(truth)))
print(classification_report(truth, preds, target_names=classes))
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix(truth, preds), display_labels=classes).plot(ax=ax, xticks_rotation=45)
plt.show()

## Step 8: Save the model and download it
Put `gesture_model.pt` in the same folder as `app.py` on your computer, then run `python app.py`.

In [ ]:
torch.save({"state_dict": model.state_dict(), "classes": classes, "pad": PAD}, "gesture_model.pt")
files.download("gesture_model.pt")

## If accuracy is low or it struggles on your webcam
- Train longer, or add more images per gesture.
- Add a few hundred crops of your own hand and room to `/content/crops/train/<gesture>/`, then re-run Steps 4-8.
- Check the confusion matrix for gestures that look alike (e.g. `like` vs `fist`) and consider dropping one.